# Task 1 — W&B backfill from a finished run (CPU only, ~5 min)

Use this when the Task 1 run finished but **W&B was not connected** during it.
Every experiment result was also saved to disk: the Optuna SQLite study, the per-epoch training history, the test tables, figures, checkpoint and ONNX model.
This notebook uploads those files to W&B with the same run names, groups and metric keys as the live logging.

Every run it creates is tagged **`backfilled`**, and each run's notes say that it was logged after the fact from saved outputs.
Say so in the report as well (one sentence in the experimental-setup section is enough).

**Setup:** Accelerator **None (CPU)** · Internet **On** · Secret `WANDB_API_KEY` attached **to this notebook** ·
*Add Input → Your Work → Notebooks →* the Task 1 notebook (its full-mode output version).

In [ ]:
import os, json, shutil, time
from pathlib import Path
import pandas as pd
import optuna, wandb

WANDB_PROJECT = "genai-assignment1"
WANDB_ENTITY = None
GROUP = "task1"
INPUT_ROOT = Path(os.environ.get("PET_INPUT_ROOT", "/kaggle/input"))
WORK = Path(os.environ.get("WORK_DIR", "/kaggle/working"))
NOTE = ("Backfilled after the fact from the saved outputs of the full Task 1 Kaggle run "
        "(the W&B secret was not attached during that run). Values are identical to the run's own files.")

if Path("/kaggle/input").exists() and not os.environ.get("WANDB_MODE"):
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

dbs = [p for p in INPUT_ROOT.rglob("task1_universal_dae.db") if p.parent.name == "task1"]   # full mode only (not task1_smoke)
assert len(dbs) == 1, f"expected exactly one full-mode Task 1 output, found: {dbs}"
ROOT = dbs[0].parent.parent.parent            # .../optuna/task1/x.db -> output root
D = {k: ROOT / k / "task1" for k in ["optuna", "results", "figures", "checkpoints", "models"]}
for k, d in D.items():
    print(f"{k:12s} {d}  ({len(list(d.glob('*'))) if d.exists() else 'MISSING'} files)")

def init(name, job_type, config=None):
    return wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, group=GROUP, name=name, job_type=job_type,
                      tags=["task1", "full", "backfilled"], notes=NOTE, config=config or {}, reinit="finish_previous")

def images(names):
    return {f"figures/{p.stem}": wandb.Image(str(p)) for p in (D["figures"] / n for n in names) if p.exists()}

def jload(name):
    p = D["results"] / name
    return json.loads(p.read_text()) if p.exists() else {}

## 1. Optuna trials: one run per trial, with the per-epoch validation score Optuna recorded

In [ ]:
db = WORK / "task1_study_copy.db"                 # inputs are read-only; SQLite needs a writable copy
shutil.copy(D["optuna"] / "task1_universal_dae.db", db)
study = optuna.load_study(study_name="task1_universal_dae", storage=f"sqlite:///{db}")
print(len(study.trials), "trials; best #", study.best_trial.number, round(study.best_value, 4))

for t in study.trials:
    run = init(f"trial-{t.number:03d}", "optuna-trial", {**t.params, **t.user_attrs, "trial": t.number})
    for ep, v in sorted(t.intermediate_values.items()):
        run.log({"epoch": ep, "val/score": v}, step=ep)
    run.summary.update({"best_score": t.value if t.value is not None else max(t.intermediate_values.values(), default=None),
                        "state": t.state.name.lower(),
                        "duration_min": t.duration.total_seconds() / 60 if t.duration else None,
                        **{k: v for k, v in t.user_attrs.items() if k.startswith("val_")}})
    run.finish()
print("trial runs logged")

In [ ]:
best = jload("optuna_best.json")
run = init("optuna-summary", "optuna-summary", {"search_space": best.get("search_space"), "n_trials": len(study.trials),
                                                 "trial_epochs": best.get("trial_epochs")})
run.summary.update({"best_value": study.best_value, "best_trial": study.best_trial.number,
                    **{f"best/{k}": v for k, v in study.best_params.items()}, "states": best.get("states")})
trials = pd.read_csv(D["results"] / "optuna_trials.csv")
run.log({"optuna_trials": wandb.Table(dataframe=trials.astype(str)),
         **images(["optuna_history.png", "optuna_importance.png", "optuna_slice.png", "optuna_parallel.png"])})
art = wandb.Artifact("task1_universal_dae-study", type="optuna-study", metadata={"backfilled": True})
art.add_file(str(D["optuna"] / "task1_universal_dae.db")); art.add_file(str(D["results"] / "optuna_trials.csv"))
run.log_artifact(art); run.finish()

## 2. Final training: per-epoch history replayed, then test results, figures, checkpoint and ONNX

In [ ]:
hp, summ, side = jload("final_hp.json"), jload("summary.json"), {}
sp = D["models"] / "task1_universal_dae.json"
if sp.exists():
    side = json.loads(sp.read_text())
cfg = {**hp, **{k: side.get(k) for k in ("latent_shape", "latent_dim", "compression_ratio", "params_M", "model_config")}}
run = init("final-training", "train", cfg)

hist = pd.read_csv(D["results"] / "training_history.csv")
for _, r in hist.iterrows():
    run.log({k: (None if pd.isna(v) else v) for k, v in r.items()}, step=int(r["epoch"]))
print(f"{len(hist)} epochs replayed")

tables = {}
for name, key in [("test_by_type.csv", "test/by_type"), ("test_by_type_severity.csv", "test/by_type_severity"),
                  ("failure_cases.csv", "test/failure_cases")]:
    p = D["results"] / name
    if p.exists():
        tables[key] = wandb.Table(dataframe=pd.read_csv(p))
figs = images(["training_curves.png", "test_representative_1.png", "test_representative_2.png", "test_failure_cases.png",
               "test_severity_curves.png", "progress_latest.png", "overfit_one_batch.png"])
run.log({**tables, **figs})

# Scalar test metrics per condition and per condition x severity (filterable/comparable in W&B)
bt = pd.read_csv(D["results"] / "test_by_type.csv", index_col=0)
bs = pd.read_csv(D["results"] / "test_by_type_severity.csv", index_col=[0, 1])
flat = {f"test/{col}/{row}": float(v) for row, s in bt.iterrows() for col, v in s.items()}
flat.update({f"test/{col}/{t}/{lv}": float(v) for (t, lv), s in bs.iterrows() for col, v in s.items()})
run.summary.update({**flat, "best_epoch": summ.get("best_epoch"), "best_val_score": summ.get("best_val_score"),
                    "onnx/max_abs_diff": summ.get("onnx_max_abs_diff"), "onnx/latency_ms": summ.get("onnx_latency_ms"),
                    "backfilled": True})

ck = D["checkpoints"] / "task1_best.pth"
if ck.exists():
    a = wandb.Artifact("task1-universal-dae-checkpoint", type="model", metadata={**hp, "backfilled": True})
    a.add_file(str(ck)); run.log_artifact(a)
ox = D["models"] / "task1_universal_dae.onnx"
if ox.exists():
    a = wandb.Artifact("task1-universal-dae-onnx", type="model", metadata={**side, "backfilled": True})
    a.add_file(str(ox)); a.add_file(str(sp)); run.log_artifact(a)
run.finish()

## 3. Skip ablation

In [ ]:
p = D["results"] / "skip_ablation.csv"
if p.exists():
    abl = pd.read_csv(p)
    for _, r in abl.iterrows():
        run = init(f"ablation-skip{int(r['skip_levels'])}", "ablation", {**hp, "skip_levels": int(r["skip_levels"])})
        run.summary.update({k: float(v) for k, v in r.items()}); run.finish()
    run = init("ablation-summary", "ablation"); run.log({"skip_ablation": wandb.Table(dataframe=abl)}); run.finish()
    print(abl.round(4))
else:
    print("skip_ablation.csv not found — the ablation did not finish in the source run")
print("\nBACKFILL COMPLETE — check the W&B project, group 'task1'.")